In [1]:
from pymilvus import (
    connections,
    FieldSchema, CollectionSchema, DataType,
    Collection
)
from pymilvus import connections




In [12]:
from pymilvus import utility

# Se a coleção já existir, dropa
if utility.has_collection("image_descriptions"):
    utility.drop_collection("image_descriptions")

# Agora cria do zero
collection = Collection("image_descriptions", schema)


In [13]:
from pymilvus import connections, Collection, CollectionSchema, FieldSchema, DataType
from sentence_transformers import SentenceTransformer
import uuid, requests
from PIL import Image, ImageEnhance
import cv2, numpy as np, easyocr
from io import BytesIO
from transformers import CLIPProcessor, CLIPModel

# ---- Conexão Milvus ----
connections.disconnect("default")
connections.connect("default", host="localhost", port="19530")

# Definição do schema
fields = [
    FieldSchema(name="id", dtype=DataType.VARCHAR, max_length=36, is_primary=True, auto_id=False),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384),  # depende do modelo
    FieldSchema(name="url", dtype=DataType.VARCHAR, max_length=500),
    FieldSchema(name="category", dtype=DataType.VARCHAR, max_length=50),
    FieldSchema(name="titles", dtype=DataType.VARCHAR, max_length=500),
    FieldSchema(name="texts", dtype=DataType.VARCHAR, max_length=2000),
]
schema = CollectionSchema(fields, description="Armazenamento de imagens processadas")



# Embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


def classify_image(image_path: str) -> str:
    labels = ["Mapa", "Gráfico", "Diagrama", "Tabela", "Outro"]

    if image_path.startswith("http"):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path).convert("RGB")

    inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True)
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)
    category = labels[probs.argmax().item()]
    return category


def extract_text_structure(image_path: str) -> dict:
    if image_path.startswith("http"):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path).convert("RGB")

    image = image.convert("L")
    image = ImageEnhance.Contrast(image).enhance(2)
    image = ImageEnhance.Sharpness(image).enhance(2)

    image_np = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    reader = easyocr.Reader(["pt"])
    result = reader.readtext(image_np)

    titles, texts = [], []
    for (bbox, text, prob) in result:
        if prob > 0.5:
            if bbox[1][1] < 50:
                titles.append(text.strip())
            else:
                texts.append(text.strip())

    return {"titles": titles, "texts": texts}


def process_and_store(image_url: str):
    try:
        category = classify_image(image_url)
        extracted = extract_text_structure(image_url)

        description = f"{category}: " + " ".join(extracted["titles"] + extracted["texts"])
        embedding = embedding_model.encode(description).tolist()

        doc_id = str(uuid.uuid4())

        # Inserção no Milvus (cada campo é uma lista, mesmo que só tenha 1 elemento)
        collection.insert([
            [doc_id],
            [embedding],
            [image_url],
            [category],
            [" ".join(extracted["titles"])],
            [" ".join(extracted["texts"])]
        ])

        collection.flush()
        print(f"[OK] Imagem armazenada no Milvus com ID {doc_id}")

    except Exception as e:
        print(f"Erro ao processar/armazenar imagem: {e}")




# Teste
process_and_store("https://smastr16.blob.core.windows.net/igeo/2012/03/mapa_aguas_subterraneas.jpg")

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
C:\Users\vinic\Documents\GitHub\Database-Chroma-RAG-Project\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


[OK] Imagem armazenada no Milvus com ID 1eeecbdd-9f83-4735-bed4-2bdb0f1fc1a4


In [23]:
def search_images(collection_name, query_text, top_k=3):
    collection = Collection(collection_name)

    # se não existir índice, cria
    if not collection.has_index():
        collection.create_index(
            field_name="embedding",
            index_params={
                "metric_type": "COSINE",
                "index_type": "IVF_FLAT",
                "params": {"nlist": 128}
            }
        )

    # carregar coleção
    collection.load()

    # Gerar embedding do texto
    query_embedding = embedding_model.encode(query_text).tolist()

    # Executar busca
    results = collection.search(
        data=[query_embedding],
        anns_field="embedding",
        param={"metric_type": "COSINE", "params": {"nprobe": 10}},
        limit=top_k,
        output_fields=["id", "url", "category", "titles", "texts"]
    )

    return results[0]


In [26]:
res = search_images("image_descriptions", "mapa de águas subterrâneas", top_k=3)
for r in res:
    print(f"ID: {r.id}")
    print(f"Score: {r.distance:.4f}")
    print(f"URL: {r.entity.get('url')}")
    print(f"Categoria: {r.entity.get('category')}")
    print(f"Títulos: {r.entity.get('titles')}")
    print(f"Texts: {r.entity.get('texts')}")
    print("-"*40)
    print(f"Distancia: {r.entity.get('distance')}")


ID: 1eeecbdd-9f83-4735-bed4-2bdb0f1fc1a4
Score: 0.7306
URL: https://smastr16.blob.core.windows.net/igeo/2012/03/mapa_aguas_subterraneas.jpg
Categoria: Mapa
Títulos: MAPA DE ÁGUAS SUBTERRÂNEAS DO ESTADO DE SÃO PAULO
Texts: #x
----------------------------------------
Distancia: 0.7306317090988159


In [27]:
def process_image_list(image_urls: list):

    for url in image_urls:
        print(f"\n[INFO] Processando imagem: {url}")
        process_and_store(url)


# 🔹 Exemplo de uso:
image_links = [
    "https://smastr16.blob.core.windows.net/igeo/2012/03/mapa_aguas_subterraneas.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/3/3f/Fronteiras_do_Brasil.png",
    "https://www.researchgate.net/publication/323517103/figure/fig1/AS:667623325425664@1536197480678/Mapa-da-area-de-estudo.png"
]

process_image_list(image_links)



[INFO] Processando imagem: https://smastr16.blob.core.windows.net/igeo/2012/03/mapa_aguas_subterraneas.jpg


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


[OK] Imagem armazenada no Milvus com ID fdb7f87c-80fc-48fa-9afa-17963379a6c8

[INFO] Processando imagem: https://upload.wikimedia.org/wikipedia/commons/3/3f/Fronteiras_do_Brasil.png
Erro ao processar/armazenar imagem: cannot identify image file <_io.BytesIO object at 0x000001BF9CF18A40>

[INFO] Processando imagem: https://www.researchgate.net/publication/323517103/figure/fig1/AS:667623325425664@1536197480678/Mapa-da-area-de-estudo.png
Erro ao processar/armazenar imagem: cannot identify image file <_io.BytesIO object at 0x000001BF9D662660>


In [11]:
# from pymilvus import connections, utility

# # Garante que está conectado (desconectando antes, só por segurança)
# connections.disconnect("default")
# connections.connect("default", host="localhost", port="19530")

# # Nome da coleção
# collection_name = "image_descriptions"

# if utility.has_collection(collection_name):
#     utility.drop_collection(collection_name)
#     print(f"Coleção '{collection_name}' excluída com sucesso.")
# else:
#     print(f"Coleção '{collection_name}' não existe.")


Coleção 'image_descriptions' excluída com sucesso.
